In [33]:
#Empezamos inicializando los parametros (fisicos, numericos, de borde y de estabilizacion) a usar en el problema. Partiendo por el problema de la caja cuadrada
#Se sigue la metodologia usada en "Numerical Simulations in Fluid Dynamics" de M. Griebel en el capitulo 3 "The Numerical Treatment of the Navier—Stokes Equations"
import numpy as np
import matplotlib.pyplot as plt
import nt

# Parámetros físicos
Lx, Ly = 1.0, 1.0   
rho = 1.0           
Re = 100.0          
nu = 1.0 / Re       

# Parámetros numéricos 
imax, jmax = 40, 40 
dx = Lx / imax
dy = Ly / jmax
dt = 0.001         

# Estabilización y convergencia para poisson
gamma = 0.9         # Factor de mezcla Upwind/Central, para cuando terminos convectivos sean dominantes a Re altos
omega = 1.5         
eps = 0.005         
iter_max = 200      


In [34]:
#inicializamos los campos de velocidad, presión y el término fuente para la ecuación de Poisson. Se usan dimensiones ligeramente mayores para acomodar las condiciones de frontera usando "celdas fantamas"

u = np.zeros((imax + 1, jmax + 2)) # vive en las paredes verticales de la malla, por eso para imax cuadros de la malla hay imax+1 caras verticales
v = np.zeros((imax + 2, jmax + 1)) # lo opuesto a u, este vive en las caras horizontales
p = np.ones((imax + 2, jmax + 2))  # +2 en ambas dimensiones para acomodar las condiciones de frontera, u y v estan alineados en el eje y e x respectivamente a p. por ello tienen dimensiones j+2 e imax+2 respectivamente
rhs_poisson = np.zeros((imax + 2, jmax + 2)) # Término fuente para la ecuación de Poisson

# Términos intermedios (Predictor)
F = np.zeros((imax + 1, jmax + 2))
G = np.zeros((imax + 2, jmax + 1))

In [35]:
# Calculamos los terminos F (u estrella) y G (v estrella) de la etapa predictor usando diferencias finitas y aplicando la forma conservativa de las derivadas parciales para los terminos convectivos. 
# Se incluyen términos de difusión y convección, con mezcla Upwind/Central para estabilidad a Re altos.

def calcular_F_G(u, v, nu, dt, dx, dy, gamma, imax, jmax):
    # calculo de F
    for i in range(1, imax):
        for j in range(1, jmax + 1):
            # Derivadas de u^2 y uv (Forma Conservativa)
            # Promedios y diferencias para el transporte de momento en X
            du2dx = (1/dx) * ( ((u[i,j] + u[i+1,j])/2)**2 + gamma * abs(u[i,j] + u[i+1,j])/2 * (u[i,j] - u[i+1,j])/2 - 
                               ((u[i-1,j] + u[i,j])/2)**2 - gamma * abs(u[i-1,j] + u[i,j])/2 * (u[i-1,j] - u[i,j])/2 )
            
            duvdy = (1/dy) * ( ((v[i,j] + v[i+1,j])/2) * ((u[i,j] + u[i,j+1])/2) + gamma * abs(v[i,j] + v[i+1,j])/2 * (u[i,j] - u[i,j+1])/2 -
                               ((v[i,j-1] + v[i+1,j-1])/2) * ((u[i,j-1] + u[i,j])/2) - gamma * abs(v[i,j-1] + v[i+1,j-1])/2 * (u[i,j-1] - u[i,j])/2 )

            # Laplaciano de u (Difusión)
            lapl_u = (u[i+1,j] - 2*u[i,j] + u[i-1,j])/dx**2 + (u[i,j+1] - 2*u[i,j] + u[i,j-1])/dy**2
            
            F[i,j] = u[i,j] + dt * (nu * lapl_u - du2dx - duvdy)
    # calculo de G
    for i in range(1, imax + 1):
        for j in range(1, jmax):
            # Derivadas de uv y v^2 (Forma Conservativa)
            duvdx = (1/dx) * ( ((u[i,j] + u[i,j+1])/2) * ((v[i,j] + v[i+1,j])/2) + gamma * abs(u[i,j] + u[i,j+1])/2 * (v[i,j] - v[i+1,j])/2 -
                               ((u[i-1,j] + u[i-1,j+1])/2) * ((v[i-1,j] + v[i,j])/2) - gamma * abs(u[i-1,j] + u[i-1,j+1])/2 * (v[i-1,j] - v[i,j])/2 )
            
            dv2dy = (1/dy) * ( ((v[i,j] + v[i,j+1])/2)**2 + gamma * abs(v[i,j] + v[i,j+1])/2 * (v[i,j] - v[i,j+1])/2 -
                               ((v[i,j-1] + v[i,j])/2)**2 - gamma * abs(v[i,j-1] + v[i,j])/2 * (v[i,j-1] - v[i,j])/2 )

            # Laplaciano de v
            lapl_v = (v[i+1,j] - 2*v[i,j] + v[i-1,j])/dx**2 + (v[i,j+1] - 2*v[i,j] + v[i,j-1])/dy**2
            
            G[i,j] = v[i,j] + dt * (nu * lapl_v - duvdx - dv2dy)
    return F, G



In [36]:
# Con este campo de velocidades "estrella" u*=(F,G) nos es simple reconocer la descomposicion de Helmholtz-Hodge a usar para resolver nuestro problema (pag 33 del libro de M. Griebel)
# en done el campo solenoidal u^(n+1) se obtiene de la resta del campo (F,G) con el gradiente de un potencial escalar (la preison p). [u^(n+1) = u* - grad(p)]
# Luego aplicando la divergencia a la descomposicion obtenemos la ecuacion de Poisson para la presion, con el lado derecho dado por la divergencia de u*=(F,G). [lapl(p) = div(F,G) = rhs_poisson]
# definimos entonces la funcion para calcular el lado derecho de la ec de poisson

def calcular_rhs_poisson(F, G, dx, dt, dy, rho, imax, jmax):
    for i in range(1, imax + 1):
        for j in range(1, jmax + 1):
            rhs_poisson[i,j] = (rho / dt) * ( (F[i,j] - F[i-1,j])/dx + (G[i,j] - G[i,j-1])/dy )
    return rhs_poisson


In [37]:
# Con el lado derecho de la ec de poisson listo, ahora escribimos el lado izq (laplaciano) mediante diferencias finitas centradas y resolvemos la ecuacion de poisson para la presion usando
# el metodo iterativo de gauss-seidel con SOR para acelerar la convergencia. Para ello debemos despejar la presion p[i,j] en terminos de sus vecinos y el lado derecho rhs_poisson[i,j]. Imponemos condiciones de borde Neumann (dp/dn = 0).

def solver_poisson_SOR(p, rhs_poisson, dx, dy, omega, eps, iter_max, imax, jmax):
   
    p_viejo = np.zeros_like(p)
    dx2 = dx**2
    dy2 = dy**2
    denominador = 2.0 * (1.0/dx2 + 1.0/dy2)
    
    for it in range(iter_max):
        # Hacemos una copia para evaluar la convergencia al final de la iteración
        p_viejo[:, :] = p[:, :]
        residuo_max = 0.0
        
        # Recorremos los nodos internos
        for i in range(1, imax + 1):
            for j in range(1, jmax + 1):
                p_nuevo = (1.0 / denominador) * ( (p[i+1, j] + p[i-1, j]) / dx2 + (p[i, j+1] + p[i, j-1]) / dy2 - rhs_poisson[i, j] )

                #Aplicamos SOR
                p[i,j] = (1.0 - omega) * p[i,j] + omega * p_nuevo
                
                # Condición de Borde de Neumann (dp/dn = 0)
                if i == 1:       p[0, j] = p[1, j]         # Pared Izquierda
                if i == imax:    p[imax+1, j] = p[imax, j] # Pared Derecha
                if j == 1:       p[i, 0] = p[i, 1]         # Pared Inferior
                if j == jmax:    p[i, jmax+1] = p[i, jmax] # Pared Superior (Tapa)
                
        # Calculamos el error máximo (Residuo) en esta iteración
        # Se calcula restando la presión anterior a la recién iterada
        for i in range(1, imax + 1):
             for j in range(1, jmax + 1):
                  error_local = abs(p[i,j] - p_viejo[i,j])
                  if error_local > residuo_max:
                      residuo_max = error_local
        
        # Chequeo de convergencia
        if residuo_max < eps:
            #print(f"Poisson convergió en {it+1} iteraciones (Residuo: {residuo_max:.6f})")# lo comento pq tapa la pantalla, pero se puede descomentar para ver la convergencia
            return p
            
    print(f"No convergió después de {iter_max} iteraciones (Residuo final: {residuo_max:.6f})")
    return p




In [38]:
# Finalmente, con la presion calculada, procedemos a actualizar los campos de velocidades solenoidales (correcion solenoidal) usando la relación u^(n+1) = u* - dt*grad(p)/rho. 
# Para esto, simplemente restamos el gradiente de la presión a los campos F y G para obtener las velocidades actualizadas u y v.

def corrector_solenoidal(u, v, F, G, p, dt, dx, dy, rho, imax, jmax):
    # Actualizamos u (Caras verticales: i=1..imax-1, j=1..jmax)
    for i in range(1, imax):
        for j in range(1, jmax + 1):
            # Gradiente de presión en X
            dp_dx = (p[i+1, j] - p[i, j]) / dx
            u[i, j] = F[i, j] - (dt / rho) * dp_dx
            
    # Actualizamos v (Caras horizontales: i=1..imax, j=1..jmax-1)
    for i in range(1, imax + 1):
        for j in range(1, jmax):
            # Gradiente de presión en Y
            dp_dy = (p[i, j+1] - p[i, j]) / dy
            v[i, j] = G[i, j] - (dt / rho) * dp_dy
            
    return u, v

In [39]:
# Con el campo de velocidades (u,v) corregido y listo para usar, planteamos las condiciones de borde para las situaciones pedidas.Partimos por las CB de la caja cuadrada, usando la condicion de no deslizamiento
# (no-slip condition, pag 30 del libro de M. Griebel), que establece 0 a las velocidades perpendiculaes a las poredes fijas y calcula el promedio de las velocidades tangenciales
# de modo que entre un paso antes y un paso despues de la pared, la velocidad tangencial sea 0. Para la tapa movil, se establece una velocidad tangencial constante (u=1) y se calcula el promedio con la velocidad del paso anterior para mantener la estabilidad.

def CB_cavidad_cuadrada(u, v, imax, jmax, u_tapa=1.0):
    # 1. Velocidades perpendiculares
    for j in range(1, jmax + 1):
        u[0, j] = 0.0          # Pared Izquierda
        u[imax, j] = 0.0       # Pared Derecha
        
    for i in range(1, imax + 1):
        v[i, 0] = 0.0          # Pared Inferior
        v[i, jmax] = 0.0       # Pared Superior
        
    # 2. Velocidades tangenciales (No slip)
    for j in range(1, jmax + 1):
        v[0, j] = -v[1, j]             # Promedio de v da 0 en la pared izquierda
        v[imax+1, j] = -v[imax, j]     # Promedio de v da 0 en la pared derecha

    for i in range(1, imax + 1):
        u[i, 0] = -u[i, 1]             # promedio de u da 0 en la pared inferior
        u[i, jmax+1] = 2.0*u_tapa - u[i, jmax] # promedio de u da u_tapa en la pared superior
        
    return u, v 

In [40]:
# Con todo listo, ahora iniciamos el ciclo principal de la simulacion (para cavidad cuadrada) en donde se siguen los pasos del metodo de proyeccion de Chorin: 1- Imponer condiciones de borde, 2- Predictor (Calcular F y G),
# 3- Calcular lado derecho de Poisson, 4- Resolver Poisson para la presion, 5- Corrector Solenoidal y obtener el campo de velocidades real y actualizado para el siguiente paso de tiempo. Se monitorea el progreso cada ciertos pasos para no llenar la pantalla.
# dado que dt=0.001 y nt=5000, la simulacion avanzara hasta t=5 segundos, lo cual es suficiente para observar el desarrollo del flujo (se puede ajustar estos parametros para simular mas o menos tiempo segun se desee)

print(f"Iniciando simulación para Re = {Re}...")

# Parámetro para saber cada cuántos pasos imprimir en pantalla
pasos_impresion = 1000
nt = 5000

for n in range(nt):
    # 1-
    u, v = CB_cavidad_cuadrada(u, v, imax, jmax, u_tapa=1.0)
    
    # 2-
    F, G = calcular_F_G(u, v, nu, dt, dx, dy, gamma, imax, jmax)
    
    # 3-
    rhs_poisson = calcular_rhs_poisson(F, G, dx, dt, dy, rho, imax, jmax)
    
    # 4-
    p = solver_poisson_SOR(p, rhs_poisson, dx, dy, omega, eps, iter_max, imax, jmax)
    
    # 5-
    u, v = corrector_solenoidal(u, v, F, G, p, dt, dx, dy, rho, imax, jmax)
    
    # Monitoreo de progreso
    if n % pasos_impresion == 0:
        print(f"Paso de tiempo {n}/{nt} completado.")

print("¡Simulación terminada!")

Iniciando simulación para Re = 100.0...
Paso de tiempo 0/5000 completado.
Paso de tiempo 1000/5000 completado.
Paso de tiempo 2000/5000 completado.
Paso de tiempo 3000/5000 completado.
Paso de tiempo 4000/5000 completado.
¡Simulación terminada!
